# GeoMapBench — simple eight-model evaluation

This notebook contains no evaluator logic. It mounts Drive, clones a frozen code release, validates the canonical 23 × 100 benchmark, runs the model suite, and displays the report. Rerun after a disconnect to resume completed IDs.

In [ ]:
# Edit only this cell, then use Runtime -> Run all.
REPO_URL = "https://github.com/asalmeskin/GeoMapBench.git"
GIT_REF = "main"  # For a paper, replace with the final v1.7.0 tag or commit SHA.

BENCHMARK_ROOT = "/content/drive/MyDrive/geomapbench_100"
RESULTS_ROOT = "/content/drive/MyDrive/geomapbench_results_final"

# None = full 100/leaf. Use 1 for the mandatory 23-call pilot, then switch to None.
PER_LEAF_LIMIT = None

MAX_COST_USD_PER_MODEL = 25.0


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import getpass, os, shutil, subprocess
from pathlib import Path

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OPENROUTER_API_KEY: ")

repo = Path("/content/GeoMapBench")
if repo.exists():
    shutil.rmtree(repo)
subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", GIT_REF], check=True)
print("checked out:", subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip())

subprocess.run(["python", "-m", "pip", "install", "-q", "-e", str(repo)], check=True)


In [ ]:
models = repo / "config/evaluation_models_2026-09.json"
output = Path(RESULTS_ROOT) / ("model_suite_full" if PER_LEAF_LIMIT is None else f"model_suite_{PER_LEAF_LIMIT}_per_leaf")

command = [
    "geomapbench-eval", "suite",
    "--benchmark-root", BENCHMARK_ROOT,
    "--models", str(models),
    "--output", str(output),
    "--max-cost-usd-per-model", str(MAX_COST_USD_PER_MODEL),
]
if PER_LEAF_LIMIT is not None:
    command += ["--per-leaf-limit", str(PER_LEAF_LIMIT)]
subprocess.run(command, check=True)


In [ ]:
import pandas as pd
report = pd.read_csv(output / "model_comparison.csv")
display(report.sort_values("macro_accuracy", ascending=False))
print("Saved to:", output)
